# Hanle Wind Speed & Direction Comparison (Real vs ERA5)

This notebook performs a comprehensive comparison of wind speed and direction between:
1. **Real Data**: Ground-based Automatic Weather Station (AWS) observations (January 2020 to July 2025).
2. **ERA5 Model Data**: Nearest grid point to Hanle site (January 2020 to December 2025).

For both datasets, it generates:
- Monthly polar wind rose scatter plots for all 12 months of all 6 years (saved separately to `data/exports`).
- A cumulative compilation polar wind rose scatter plot for all available data combined (saved to `data/exports`).

In [1]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import gc

# Set plot styling
plt.style.use('default')
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Constants
HANLE_LAT = 32.779
HANLE_LON = 78.964

ERA5_DIR = r"../../data/raw/HN Data Monthly"
REAL_DIR = r"../../data/processed/hanle_published"
OUTPUT_DIR = r"../../data/exports/hanle_monthly_roses"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment and configuration loaded! [OK]")

Environment and configuration loaded! [OK]


In [2]:
# ============================================================
# 2. LOAD ERA5 DATA FOR HANLE SITE (2020-2025)
# ============================================================
print("Loading ERA5 NetCDF files for Hanle site...")
era5_files = sorted(glob.glob(os.path.join(ERA5_DIR, "india_*.nc")))
print(f"Found {len(era5_files)} NetCDF files.")

era5_records = []
for f in era5_files:
    with xr.open_dataset(f) as ds:
        # Select nearest grid point to Hanle site
        point_ds = ds.sel(latitude=HANLE_LAT, longitude=HANLE_LON, method='nearest')
        u10 = point_ds['u10'].values
        v10 = point_ds['v10'].values
        valid_time = pd.to_datetime(point_ds['valid_time'].values)
        
        # Wind Speed and Meteorological Wind Direction (from which wind is blowing)
        speed = np.sqrt(u10**2 + v10**2)
        # 270 - arctan2(v, u) converted to degrees
        direction = (270 - np.degrees(np.arctan2(v10, u10))) % 360
        
        df_pt = pd.DataFrame({
            'datetime': valid_time,
            'wind_speed': speed,
            'wind_direction': direction
        })
        era5_records.append(df_pt)

df_era5 = pd.concat(era5_records, ignore_index=True)
df_era5['year'] = df_era5['datetime'].dt.year
df_era5['month'] = df_era5['datetime'].dt.month

print(f"Loaded {len(df_era5):,} ERA5 hourly records.")
print(df_era5.head())

Loading ERA5 NetCDF files for Hanle site...
Found 72 NetCDF files.


Loaded 52,608 ERA5 hourly records.
             datetime  wind_speed  wind_direction  year  month
0 2020-01-01 00:00:00    1.872612      178.681458  2020      1
1 2020-01-01 01:00:00    1.791171      177.322311  2020      1
2 2020-01-01 02:00:00    2.000545      178.131439  2020      1
3 2020-01-01 03:00:00    2.160432      177.344421  2020      1
4 2020-01-01 04:00:00    1.941265      178.700165  2020      1


In [3]:
# ============================================================
# 3. LOAD REAL GROUND-BASED DATA FOR HANLE (2020-2025)
# ============================================================
print("Loading and downsampling Real AWS observations (100x reduction for efficiency)...")
real_files = sorted(glob.glob(os.path.join(REAL_DIR, "hanle_published_*.csv")))
print(f"Found {len(real_files)} CSV files.")

real_records = []
for f in real_files:
    df_m = pd.read_csv(f, usecols=['datetime_ist', 'wind_speed', 'wind_direction'])
    # Take every 100th record
    df_ds = df_m.iloc[::100].copy()
    real_records.append(df_ds)
    del df_m
    gc.collect()

df_real = pd.concat(real_records, ignore_index=True)
df_real['datetime_ist'] = pd.to_datetime(df_real['datetime_ist'])
df_real['year'] = df_real['datetime_ist'].dt.year
df_real['month'] = df_real['datetime_ist'].dt.month

print(f"Loaded {len(df_real):,} downsampled Real records.")
print(df_real.head())

Loading and downsampling Real AWS observations (100x reduction for efficiency)...
Found 67 CSV files.


Loaded 334,309 downsampled Real records.
         datetime_ist  wind_speed  wind_direction  year  month
0 2020-01-01 17:19:32        15.6           214.0  2020      1
1 2020-01-01 17:30:07        12.5           211.0  2020      1
2 2020-01-01 17:38:27        11.0           214.0  2020      1
3 2020-01-01 17:46:47        16.0           211.0  2020      1
4 2020-01-01 17:55:07        14.7           223.0  2020      1


In [4]:
# ============================================================
# 4. PLOTTING AND SAVING HELPER FUNCTIONS
# ============================================================
def make_filename(site_name, years, months, mode):
    site_name = site_name.replace(" ", "_")
    year_str = "-".join(years) if isinstance(years, list) else str(years)
    month_str = "-".join(months) if isinstance(months, list) else str(months)
    return f"{site_name}_wind_{year_str}_{month_str}_{mode}.png"

def save_fig(filename, folder=OUTPUT_DIR, dpi=300):
    path = os.path.join(folder, filename)
    fig = plt.gcf()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure -> {path}")

def plot_polar_scatter(speeds, directions, title, site_name, years, months, mode, max_speed=30.0):
    if len(speeds) == 0:
        print(f"No data to plot for {title}")
        return
        
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'polar': True})
    
    theta = np.radians(directions)
    sc = ax.scatter(theta, speeds, c=speeds, cmap='viridis', vmin=0, vmax=max_speed, s=3, alpha=0.3, edgecolors='none')
    
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)  # Clockwise
    ax.set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315], ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_rlabel_position(45)
    
    mean_val = np.mean(speeds)
    max_val = np.max(speeds)
    ax.set_title(f"{title}\nn={len(speeds):,} | mean={mean_val:.2f} m/s | max={max_val:.2f} m/s", 
                 pad=25, fontsize=10, fontweight='bold')
    
    cbar = fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.08, shrink=0.7)
    cbar.set_label("Wind Speed (m/s)", fontsize=9)
    
    filename = make_filename(site_name, years, months, mode)
    save_fig(filename)
    plt.close(fig)

In [5]:
# ============================================================
# 5. GENERATE PLOTS FOR ERA5 DATA
# ============================================================
print("Generating ERA5 monthly plots (2020-2025)...")
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June', 
               7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}

years_list = sorted(df_era5['year'].unique())
months_list = range(1, 13)

# Generate monthly separate plots
for y in years_list:
    for m in months_list:
        df_filtered = df_era5[(df_era5['year'] == y) & (df_era5['month'] == m)]
        title = f"Hanle ERA5 Wind Rose - {month_names[m]} {y}"
        plot_polar_scatter(
            speeds=df_filtered['wind_speed'],
            directions=df_filtered['wind_direction'],
            title=title,
            site_name="Hanle_ERA5",
            years=[str(y)],
            months=[f"{m:02d}"],
            mode="separate"
        )

# Generate combined plot
print("Generating ERA5 combined compilation plot...")
plot_polar_scatter(
    speeds=df_era5['wind_speed'],
    directions=df_era5['wind_direction'],
    title="Hanle ERA5 Wind Rose - Combined (2020-2025)",
    site_name="Hanle_ERA5",
    years=["2020-2025"],
    months=["all"],
    mode="combined"
)

Generating ERA5 monthly plots (2020-2025)...


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020_01_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020_02_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020_03_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020_04_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020_05_separate.png


... (64 outputs left out)

Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2025_10_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2025_11_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2025_12_separate.png
Generating ERA5 combined compilation plot...


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_ERA5_wind_2020-2025_all_combined.png


In [6]:
# ============================================================
# 6. GENERATE PLOTS FOR REAL (AWS) DATA
# ============================================================
print("Generating Real AWS monthly plots (2020-2025)...")

years_list = sorted(df_real['year'].unique())
months_list = range(1, 13)

# Generate monthly separate plots
for y in years_list:
    for m in months_list:
        df_filtered = df_real[(df_real['year'] == y) & (df_real['month'] == m)]
        if len(df_filtered) == 0:
            print(f"Skipping {month_names[m]} {y} (No real data available)")
            continue
        title = f"Hanle Real Wind Rose - {month_names[m]} {y}"
        plot_polar_scatter(
            speeds=df_filtered['wind_speed'],
            directions=df_filtered['wind_direction'],
            title=title,
            site_name="Hanle_Real",
            years=[str(y)],
            months=[f"{m:02d}"],
            mode="separate"
        )

# Generate combined plot
print("Generating Real combined compilation plot...")
plot_polar_scatter(
    speeds=df_real['wind_speed'],
    directions=df_real['wind_direction'],
    title="Hanle Real Wind Rose - Combined (2020-2025)",
    site_name="Hanle_Real",
    years=["2020-2025"],
    months=["all"],
    mode="combined"
)

Generating Real AWS monthly plots (2020-2025)...


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020_01_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020_02_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020_03_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020_04_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020_05_separate.png


... (59 outputs left out)

Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2025_05_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2025_06_separate.png


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2025_07_separate.png
Skipping August 2025 (No real data available)
Skipping September 2025 (No real data available)
Skipping October 2025 (No real data available)
Skipping November 2025 (No real data available)
Skipping December 2025 (No real data available)
Generating Real combined compilation plot...


Saved figure -> ../../data/exports/hanle_monthly_roses\Hanle_Real_wind_2020-2025_all_combined.png
